# Telco Customer Churn – Exploratory Data Analysis (EDA)

In this notebook, I’m exploring the **Telco Customer Churn dataset** to understand which customer attributes drive churn
(i.e., customers leaving the service). The goal is to generate business insights that can help reduce churn and improve
customer retention.

Churn is one of the most important metrics for any subscription-based business – acquiring new customers is much more
expensive than retaining existing ones. Through EDA, I want to identify patterns such as:

- Which customer segments have the highest churn rate?
- Do contract type, tenure, or monthly charges influence churn?
- Are there early warning indicators we can use to intervene?

---

## Dataset Description

This dataset contains information about Telco's customers, their demographics, services they subscribed to,
billing details, and whether they churned or not. Here's a quick explanation of each column in plain English:

| Column | Description |
|-------|-------------|
| **customerID** | Unique identifier for each customer (not predictive). |
| **gender** | Customer's gender (Male, Female). |
| **SeniorCitizen** | Indicates if the customer is a senior citizen (1 = Yes, 0 = No). |
| **Partner** | Whether the customer has a partner (Yes/No). |
| **Dependents** | Whether the customer has dependents (Yes/No). |
| **tenure** | Number of months the customer has stayed with the company. |
| **PhoneService** | Whether the customer has a phone service (Yes/No). |
| **MultipleLines** | Whether the customer has multiple phone lines (Yes/No/No phone service). |
| **InternetService** | Type of internet service (DSL/Fiber optic/No). |
| **OnlineSecurity** | Whether the customer has online security add-on (Yes/No/No internet). |
| **OnlineBackup** | Whether the customer has online backup add-on (Yes/No/No internet). |
| **DeviceProtection** | Whether the customer has device protection add-on (Yes/No/No internet). |
| **TechSupport** | Whether the customer has tech support add-on (Yes/No/No internet). |
| **StreamingTV** | Whether the customer has streaming TV service (Yes/No/No internet). |
| **StreamingMovies** | Whether the customer has streaming movies service (Yes/No/No internet). |
| **Contract** | Type of contract (Month-to-month/One year/Two year). |
| **PaperlessBilling** | Whether the customer is on paperless billing (Yes/No). |
| **PaymentMethod** | Customer’s payment method (Electronic check, Mailed check, Bank transfer, Credit card). |
| **MonthlyCharges** | Amount charged per month (numeric). |
| **TotalCharges** | Total amount charged to the customer (numeric – may have missing or blank values). |
| **Churn** | Target variable: Yes = customer left, No = customer stayed. |

---

### Hypotheses & Business Questions

Before diving into EDA, here are some things I expect to find (hypotheses):

- **Contract type:** Customers with month-to-month contracts should churn more compared to yearly contracts.
- **Tenure:** Newer customers (low tenure) may churn at a higher rate (onboarding issue?).
- **MonthlyCharges:** Very high charges might push customers to leave (price sensitivity).
- **Senior citizens:** May have different churn patterns compared to younger customers.
- **Add-on services:** Customers with more add-ons might be "stickier" and less likely to churn.
- **Payment method:** Electronic check users might churn more if payments fail or feel inconvenient.

I’ll validate these hypotheses with visualizations and numbers in the next steps.


##  My Plan

I will:
- Handle missing values step by step (with some common sense, not just blindly dropping)
- Explore each feature individually (univariate analysis)
- Explore how each feature relates to survival (bivariate analysis)
- Engineer new features.
- Transform skewed data for better model performance
- Prepare a clean, numeric, model-ready dataset


In [ ]:
from google.colab import files
files.upload()  # Upload your kaggle.json file here

In [3]:
!mkdir -p ~/.kaggle
!cp "kaggle (2).json" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download -d blastchar/telco-customer-churn

Dataset URL: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
License(s): copyright-authors
  0% 0.00/172k [00:00<?, ?B/s]
100% 172k/172k [00:00<00:00, 398MB/s]


In [6]:
!unzip telco-customer-churn.zip

Archive:  telco-customer-churn.zip
  inflating: WA_Fn-UseC_-Telco-Customer-Churn.csv  


In [7]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

In [8]:
# Load datset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [20]:
print(f"Dataset have {df.shape[0]} rows and {df.shape[1]} columns")

Dataset have 7043 rows and 21 columns


In [11]:
# quick info of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [18]:
# Quick decriptive statistic for numerical cols
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


In [19]:
# Quick decriptive statistic for categorical cols
df.describe(include = 'object').T

,count,unique,top,freq
customerID,7043,7043,3186-AJIEK,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


### Observations so far

- The dataset has `(7043 rows and 21 columns)`.  
- `customerID` is unique for each customer → not useful for prediction.  
- `TotalCharges` appears as `object` → may have missing or blank values; needs cleaning.  
- Other numeric columns: `tenure`, `MonthlyCharges`, `TotalCharges`.  
- Many categorical columns: `gender`, `Partner`, `Dependents`, `PhoneService`, `Contract`, `PaymentMethod`, etc.  
- Now moving toward **missing** and **inconsistent** data .  